# 03 · 10bps 风险容量反解

对每个模型族与 quantile candidate 独立生成 **45-point** action grid（`x_adv=0` + 44 个正档），分别对 total risk 和 impact guardrail 沿 `x_adv` 做 `cummax`，再反解 10bps 容量。跨 quantile 不排序。

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'outputs' / '02_stage_contract.json').exists():
            return candidate
    raise FileNotFoundError('run 02_quantile_risk_model.ipynb first')


def quantile_label(quantile: float) -> str:
    return f'q{int(round(100 * quantile)):02d}'


ROOT = find_repo_root()
OUTPUT_DIR = ROOT / 'outputs'
ACTIVE = json.loads((OUTPUT_DIR / '00_active_data.json').read_text(encoding='utf-8'))
STAGE_01 = json.loads((OUTPUT_DIR / '01_stage_contract.json').read_text(encoding='utf-8'))
STAGE_02 = json.loads((OUTPUT_DIR / '02_stage_contract.json').read_text(encoding='utf-8'))
PUBLIC_MANIFEST = json.loads((ROOT / 'data' / 'data_manifest.json').read_text(encoding='utf-8'))
BUNDLE = joblib.load(ROOT / STAGE_02['model_bundle'])
RUN_MODE = STAGE_02['run_mode']
RISK_BUDGET_BPS = 10.0
CAPACITY_RISK_MEASURE = 'absolute_final_quantile'
SHAPE_H_MODE = str(BUNDLE['shape_h_mode'])
CONTINUOUS_PREDICTION_SOURCE = 'quantile_specific_H_piecewise_linear'
impact_guardrail_quantile = float(BUNDLE['impact_guardrail_quantile'])
QUANTILE_REGISTRY = [float(value) for value in BUNDLE['quantile_registry']]
ACTION_GRID = np.array([0.0, *PUBLIC_MANIFEST['cash_adv_grid']], dtype=float)
if len(ACTION_GRID) != 45 or not np.all(np.diff(ACTION_GRID) > 0):
    raise ValueError('ACTION_GRID must be a strictly increasing 45-point grid')


In [ ]:
anchors = pd.read_parquet(ROOT / STAGE_01['capacity_anchor_panel'])
b3 = pd.read_parquet(ROOT / ACTIVE['files']['b3'])
b1 = pd.read_parquet(ROOT / ACTIVE['files']['b1'])
fill = pd.read_parquet(ROOT / ACTIVE['files']['fill'])

for frame in [anchors, b3, b1, fill]:
    frame['date'] = pd.to_datetime(frame['date'], errors='coerce').dt.normalize()
    frame['sym'] = frame['sym'].astype(str)
anchors['side'] = anchors['side'].astype(str).str.lower().str.strip()
anchors['quote_strategy'] = anchors['quote_strategy'].astype(str).str.lower().str.strip()
b3['side'] = b3['side'].astype(str).str.lower().str.strip()
b3['quote_strategy'] = b3['quote_strategy'].astype(str).str.lower().str.strip()
b1['side'] = b1['side'].astype(str).str.lower().str.strip()
fill['shock_side'] = fill['shock_side'].astype(str).str.lower().str.strip()

key_columns = ['date', 'sym', 'side', 'quote_strategy']
b3_keys = b3[key_columns].drop_duplicates()
evaluation_anchors = b3_keys.merge(anchors, on=key_columns, how='left', indicator=True)
anchor_coverage = pd.DataFrame([{
    'run_mode': RUN_MODE, 'n_b3_keys': len(b3_keys),
    'n_anchor_matches': int(evaluation_anchors['_merge'].eq('both').sum()),
    'anchor_coverage_rate': float(evaluation_anchors['_merge'].eq('both').mean()),
}])
anchor_coverage.to_csv(OUTPUT_DIR / '03_capacity_anchor_coverage.csv', index=False)
evaluation_anchors = evaluation_anchors[evaluation_anchors['_merge'].eq('both')].drop(columns=['_merge']).reset_index(drop=True)
if evaluation_anchors.empty:
    raise ValueError('no B3 anchors matched the model panel')

fill_key = fill[['date', 'sym', 'shock_side', 'x_safe_fill']].rename(columns={'shock_side': 'side'}).drop_duplicates(['date', 'sym', 'side'])
b1_cash = b1[['date', 'sym', 'side', 'adv_cash']].drop_duplicates(['date', 'sym', 'side']) if 'adv_cash' in b1.columns else pd.DataFrame(columns=['date', 'sym', 'side', 'adv_cash'])
evaluation_anchors = evaluation_anchors.merge(fill_key, on=['date', 'sym', 'side'], how='left')
evaluation_anchors = evaluation_anchors.merge(b1_cash, on=['date', 'sym', 'side'], how='left')
evaluation_anchors['anchor_id'] = np.arange(len(evaluation_anchors), dtype=np.int64)


In [ ]:
feature_columns = list(BUNDLE['feature_columns'])
condition_features = list(BUNDLE['condition_features'])
medians = dict(BUNDLE['medians'])
calibration_table = BUNDLE['calibration_table'].copy()


def ratio_bucket(values: pd.Series) -> pd.Series:
    bins = [-np.inf, 0.001, 0.0025, 0.005, 0.01, 0.02, 0.05, np.inf]
    labels = ['<=0.1%', '0.1-0.25%', '0.25-0.5%', '0.5-1%', '1-2%', '2-5%', '>5%']
    return pd.cut(pd.to_numeric(values, errors='coerce'), bins=bins, labels=labels).astype(str)


def make_matrix(frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    matrix = frame.reindex(columns=columns).apply(pd.to_numeric, errors='coerce')
    for column in columns:
        matrix[column] = matrix[column].fillna(float(medians.get(column, 0.0)))
    return matrix.astype('float32')


def interpolate_quantile_shape(frame: pd.DataFrame, h_grid: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    values = np.zeros(len(frame), dtype=float)
    shape_h_support_status = np.full(len(frame), 'missing_side', dtype=object)
    sides = frame['side'].astype(str).str.lower().to_numpy()
    x_values = pd.to_numeric(frame['x_adv'], errors='coerce').to_numpy(dtype=float)
    for side, side_grid in h_grid.groupby('side', sort=False):
        positions = np.flatnonzero(sides == side)
        if len(positions) == 0:
            continue
        side_grid = side_grid.sort_values('x_adv')
        support_x = side_grid['x_adv'].to_numpy(dtype=float)
        support_h = side_grid['h_value'].to_numpy(dtype=float)
        values[positions] = np.interp(x_values[positions], support_x, support_h, left=support_h[0], right=support_h[-1])
        shape_h_support_status[positions] = np.where(x_values[positions] > support_x[-1] + 1e-12, 'right_clipped', 'in_support')
    return values, shape_h_support_status


def final_buffer_series(frame: pd.DataFrame, family: str, quantile: float) -> np.ndarray:
    table = calibration_table[
        calibration_table['model_family'].eq(family)
        & np.isclose(calibration_table['quantile_level'], quantile)
    ].copy()
    fallback = float(table['final_buffer_bps'].median()) if len(table) else 0.0
    mapping = table.set_index(['side', 'ratio_bucket'])['final_buffer_bps'].to_dict() if len(table) else {}
    return np.array([float(mapping.get((side, bucket), fallback)) for side, bucket in zip(frame['side'], frame['ratio_bucket'])])


def invert_monotone_boundary(x_values: np.ndarray, risk_values: np.ndarray, budget_bps: float, interpolate: bool) -> dict:
    x_values = np.asarray(x_values, dtype=float)
    risk_values = np.asarray(risk_values, dtype=float)
    valid = np.isfinite(x_values) & np.isfinite(risk_values)
    x_values, risk_values = x_values[valid], risk_values[valid]
    if len(x_values) == 0:
        return {'x_safe': np.nan, 'status': 'invalid_curve'}
    if risk_values[0] > budget_bps:
        return {'x_safe': 0.0, 'status': 'zero_capacity_no_order_risk_exceeds_budget'}
    feasible = np.flatnonzero(risk_values <= budget_bps)
    if len(feasible) == len(risk_values):
        return {'x_safe': float(x_values[-1]), 'status': 'all_grid_points_feasible'}
    lower_index = int(feasible[-1])
    upper_index = lower_index + 1
    if not interpolate:
        return {'x_safe': float(x_values[lower_index]), 'status': 'grid_floor_crossing'}
    x_low, x_high = x_values[lower_index], x_values[upper_index]
    risk_low, risk_high = risk_values[lower_index], risk_values[upper_index]
    if risk_high <= risk_low + 1e-12:
        return {'x_safe': float(x_low), 'status': 'flat_crossing_grid_floor'}
    fraction = np.clip((budget_bps - risk_low) / (risk_high - risk_low), 0.0, 1.0)
    return {'x_safe': float(x_low + fraction * (x_high - x_low)), 'status': 'crossing_interpolated'}


def capacity_candidate_row(group: pd.DataFrame, family: str, quantile: float, solver: str, interpolate: bool) -> dict:
    ordered = group.sort_values('x_adv').copy()
    combined_risk = np.maximum(ordered['total_risk_final_monotone'], ordered['impact_risk_final_monotone'])
    boundary = invert_monotone_boundary(ordered['x_adv'].to_numpy(), combined_risk.to_numpy(), RISK_BUDGET_BPS, interpolate)
    first = ordered.iloc[0]
    x_safe_price_risk = boundary['x_safe']
    h_support_status = 'right_clipped' if ordered['h_support_status'].eq('right_clipped').any() else str(ordered['h_support_status'].iloc[-1])
    support_cap_status = 'not_applicable' if family == 'direct_lgbm' else 'within_train_support'
    if family == 'shape_strength' and np.isfinite(x_safe_price_risk):
        supported_x = ordered.loc[ordered['h_support_status'].eq('in_support'), 'x_adv']
        if len(supported_x) and x_safe_price_risk > float(supported_x.max()) + 1e-12:
            x_safe_price_risk = float(supported_x.max())
            boundary['status'] = 'support_cap_reached'
            support_cap_status = 'support_cap_reached'
    x_safe_fill = pd.to_numeric(pd.Series([first.get('x_safe_fill')]), errors='coerce').iloc[0]
    x_safe_final = min(x_safe_price_risk, float(x_safe_fill)) if np.isfinite(x_safe_price_risk) and np.isfinite(x_safe_fill) else np.nan
    adv_cash = pd.to_numeric(pd.Series([first.get('adv_cash')]), errors='coerce').iloc[0]
    return {
        'date': first['date'], 'sym': first['sym'], 'side': first['side'], 'quote_strategy': first['quote_strategy'],
        'anchor_id': int(first['anchor_id']), 'model_family': family, 'quantile_level': quantile,
        'prediction_stage': 'final', 'calibration_variant': 'calibration_plus_causal_floor',
        'floor_variant': 'causal_history_only', 'boundary_solver': solver,
        'candidate_policy_id': f'{family}__{quantile_label(quantile)}__final__calibration_plus_causal_floor__{solver}',
        'selection_eligible_solver': solver == 'monotone_linear_interp',
        'risk_budget_bps': RISK_BUDGET_BPS, 'impact_guardrail_quantile': impact_guardrail_quantile,
        'capacity_risk_measure': CAPACITY_RISK_MEASURE, 'shape_h_mode': SHAPE_H_MODE,
        'continuous_prediction_source': CONTINUOUS_PREDICTION_SOURCE if family == 'shape_strength' else 'direct_lgbm_grid_piecewise_linear',
        'h_support_status': h_support_status, 'support_cap_status': support_cap_status,
        'x_safe_price_risk': x_safe_price_risk, 'x_safe_fill': x_safe_fill, 'x_safe_final': x_safe_final,
        'adv_cash': adv_cash,
        'submitted_cash_price_only': x_safe_price_risk * adv_cash if np.isfinite(x_safe_price_risk) and np.isfinite(adv_cash) else np.nan,
        'submitted_cash_final_with_fill': x_safe_final * adv_cash if np.isfinite(x_safe_final) and np.isfinite(adv_cash) else np.nan,
        'capacity_status': boundary['status'], 'observed_grid_count': int(first['observed_grid_count']),
        'scored_grid_count': len(ordered), 'grid_source': 'manifest_44_positive_plus_model_no_order_anchor',
    }


In [ ]:
capacity_rows = []
monotonicity_rows = []
curve_samples = []
batch_size = 250 if RUN_MODE == 'smoke' else 1000
observed_panel = pd.read_parquet(ROOT / STAGE_01['model_panel'], columns=['date', 'sym', 'side', 'quote_strategy', 'x_adv'])
observed_panel['date'] = pd.to_datetime(observed_panel['date'], errors='coerce').dt.normalize()
observed_counts = observed_panel.groupby(key_columns)['x_adv'].nunique().rename('observed_grid_count').reset_index()
evaluation_anchors = evaluation_anchors.merge(observed_counts, on=key_columns, how='left')
evaluation_anchors['observed_grid_count'] = evaluation_anchors['observed_grid_count'].fillna(0).astype(int)

for batch_start in range(0, len(evaluation_anchors), batch_size):
    anchor_batch = evaluation_anchors.iloc[batch_start:batch_start + batch_size].copy()
    grid = anchor_batch.loc[anchor_batch.index.repeat(len(ACTION_GRID))].copy().reset_index(drop=True)
    grid['x_adv'] = np.tile(ACTION_GRID, len(anchor_batch))
    grid['ratio_bucket'] = ratio_bucket(grid['x_adv'])
    X_grid = make_matrix(grid, feature_columns)
    grid_x0 = grid.copy()
    grid_x0['x_adv'] = 0.0
    X_grid_x0 = make_matrix(grid_x0, feature_columns)
    X_condition = make_matrix(grid, condition_features)
    impact_raw = np.maximum(BUNDLE['impact_model'].predict(X_grid), 0.0)
    impact_raw[grid['x_adv'].eq(0.0).to_numpy()] = 0.0
    grid['impact_risk_final'] = impact_raw + float(BUNDLE['impact_buffer_bps'])
    grid.loc[grid['x_adv'].eq(0.0), 'impact_risk_final'] = 0.0

    for family in ['direct_lgbm', 'shape_strength']:
        for quantile in QUANTILE_REGISTRY:
            if family == 'direct_lgbm':
                total_raw = np.maximum(BUNDLE['direct_models'][quantile].predict(X_grid), 0.0)
                h_support_status = np.full(len(grid), 'not_applicable', dtype=object)
            else:
                h_grid = BUNDLE['shape_grids'][quantile]
                h_values, h_support_status = interpolate_quantile_shape(grid, h_grid)
                base_prediction_at_x0 = np.maximum(BUNDLE['direct_models'][quantile].predict(X_grid_x0), 0.0)
                strength = np.maximum(BUNDLE['shape_strength_models'][quantile].predict(X_condition), 0.0)
                total_raw = np.maximum(base_prediction_at_x0 + strength * h_values, 0.0)
            candidate = grid[key_columns + ['anchor_id', 'x_adv', 'x_safe_fill', 'adv_cash', 'observed_grid_count', 'impact_risk_final']].copy()
            candidate['h_support_status'] = h_support_status
            candidate['shape_h_mode'] = SHAPE_H_MODE
            candidate['capacity_risk_measure'] = CAPACITY_RISK_MEASURE
            candidate['continuous_prediction_source'] = CONTINUOUS_PREDICTION_SOURCE if family == 'shape_strength' else 'direct_lgbm_grid_piecewise_linear'
            candidate['total_risk_final'] = total_raw + final_buffer_series(candidate.assign(ratio_bucket=grid['ratio_bucket']), family, quantile)
            candidate['total_drop_raw'] = candidate.groupby('anchor_id')['total_risk_final'].diff()
            candidate['impact_drop_raw'] = candidate.groupby('anchor_id')['impact_risk_final'].diff()
            candidate['total_risk_final_monotone'] = candidate.groupby('anchor_id')['total_risk_final'].cummax()
            candidate['impact_risk_final_monotone'] = candidate.groupby('anchor_id')['impact_risk_final'].cummax()
            for anchor_id, group in candidate.groupby('anchor_id', sort=False):
                monotonicity_rows.append({
                    'anchor_id': int(anchor_id), 'model_family': family, 'quantile_level': quantile,
                    'raw_total_drop_count': int((group['total_drop_raw'] < -1e-12).sum()),
                    'raw_impact_drop_count': int((group['impact_drop_raw'] < -1e-12).sum()),
                    'postprocess_total_drop_count': int((group['total_risk_final_monotone'].diff() < -1e-12).sum()),
                    'postprocess_impact_drop_count': int((group['impact_risk_final_monotone'].diff() < -1e-12).sum()),
                    'ratio_curve_monotone': True,
                })
                if family == 'direct_lgbm':
                    solvers = [('monotone_linear_interp', True), ('grid_floor_diagnostic', False)]
                else:
                    solvers = [
                        ('monotone_linear_interp', True),
                        ('exact_h_piecewise_linear_diagnostic', True),
                        ('shape_grid_floor_diagnostic', False),
                    ]
                for solver, interpolate in solvers:
                    capacity_rows.append(capacity_candidate_row(group, family, quantile, solver, interpolate))
            if batch_start == 0:
                sample_ids = set(anchor_batch['anchor_id'].head(8).tolist())
                sample = candidate[candidate['anchor_id'].isin(sample_ids)].copy()
                sample['model_family'] = family
                sample['quantile_level'] = quantile
                curve_samples.append(sample)

capacity_candidates = pd.DataFrame(capacity_rows)
monotonicity = pd.DataFrame(monotonicity_rows)
curve_sample = pd.concat(curve_samples, ignore_index=True) if curve_samples else pd.DataFrame()
if not monotonicity[['postprocess_total_drop_count', 'postprocess_impact_drop_count']].eq(0).all().all():
    raise ValueError('postprocessed ratio curve is not monotone')
capacity_candidates.to_parquet(OUTPUT_DIR / '03_capacity_candidates.parquet', index=False, compression='zstd')
monotonicity.to_csv(OUTPUT_DIR / '03_monotonicity_diagnostics.csv', index=False)
curve_sample.to_parquet(OUTPUT_DIR / '03_capacity_curve_sample.parquet', index=False, compression='zstd')


In [ ]:
stage_contract = {
    'run_mode': RUN_MODE, 'evaluation_scope': STAGE_02['evaluation_scope'],
    'risk_budget_bps': RISK_BUDGET_BPS, 'impact_guardrail_quantile': impact_guardrail_quantile,
    'capacity_risk_measure': CAPACITY_RISK_MEASURE, 'shape_h_mode': SHAPE_H_MODE,
    'continuous_prediction_source': CONTINUOUS_PREDICTION_SOURCE,
    'h_support_status': ['in_support', 'right_clipped', 'missing_side', 'not_applicable'],
    'action_grid_count': len(ACTION_GRID), 'action_grid': ACTION_GRID.tolist(),
    'cross_quantile_order_enforced': False, 'ratio_curve_monotone_postprocess': 'cummax',
    'capacity_candidates': 'outputs/03_capacity_candidates.parquet',
    'monotonicity_diagnostics': 'outputs/03_monotonicity_diagnostics.csv',
    'b3_boundary_missing_row_policy': 'excluded_from_risk_and_cash_denominators',
}
(OUTPUT_DIR / '03_stage_contract.json').write_text(json.dumps(stage_contract, ensure_ascii=False, indent=2), encoding='utf-8')
print(capacity_candidates.groupby(['model_family', 'quantile_level', 'boundary_solver'])['x_safe_price_risk'].agg(['count', 'mean', 'median']).to_string())
print(anchor_coverage.to_string(index=False))
